<a href="https://colab.research.google.com/github/Kaiking28/ECON3916-Statistical-Machine-Learning/blob/main/lab13/lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# OLS via normal equations:
def ols(X, y):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    return beta, resid

def add_const(arr):
    return np.column_stack([np.ones(len(arr)), arr])

df = pd.read_csv('Zillow_California_2026_Hedonic.csv')
y, age, dist = df['Sale_Price'].values, df['Property_Age'].values, df['Distance_to_Tech_Hub'].values

# Step 1: Naive bivariate
naive_model, _ = ols(add_const(age), y)
print("Naive Age Coefficient:", naive_model[1])

# Step 2: Multivariate
multi_model, _ = ols(np.column_stack([np.ones(len(y)), age, dist]), y)
print("Multivariate Age Coefficient:", multi_model[1])

# Step 3: FWL three residual extractions
_, price_resid = ols(add_const(dist), y)
_, age_resid   = ols(add_const(dist), age)
fwl_model, _   = ols(age_resid.reshape(-1,1), price_resid)
print("FWL Isolated Age Coefficient:", fwl_model[0])

Naive Age Coefficient: 5573.630595439931
Multivariate Age Coefficient: -2063.129216802132
FWL Isolated Age Coefficient: -2063.1292168021396


In [3]:
# ============================================================
#  P.R.I.M.E. EXPANSION — Interactive 3D Regression Hyperplane
#  California Hedonic Pricing Model · 2026
# ============================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.graph_objects as go
from google.colab import files
import io

# ── Load Data ────────────────────────────────────────────────
uploaded = files.upload()                        # upload your CSV when prompted
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

# ── Fit the Multivariate OLS Model ───────────────────────────
# This is the same model from Step 2 of your FWL lab
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()

# Extract the three coefficients from the fitted model object:
#   params['Intercept']            → β₀ (where the plane crosses the y-axis)
#   params['Property_Age']         → β₁ (slope along the age axis)
#   params['Distance_to_Tech_Hub'] → β₂ (slope along the distance axis)
b0 = multi_model.params['Intercept']
b1 = multi_model.params['Property_Age']
b2 = multi_model.params['Distance_to_Tech_Hub']

print(f"β₀ Intercept            : {b0:>12,.2f}")
print(f"β₁ Property_Age         : {b1:>12,.4f}  $/yr")
print(f"β₂ Distance_to_Tech_Hub : {b2:>12,.4f}  $/mile")
print(f"R²                      : {multi_model.rsquared:.4f}")

# ── Build the Meshgrid for the Regression Surface ────────────
# np.linspace creates 60 evenly-spaced values spanning each variable's
# observed range. np.meshgrid then expands them into a 60×60 grid of
# (Age, Distance) coordinate pairs — every combination of both axes.
# This grid IS the "canvas" onto which we paint the regression plane.
age_range  = np.linspace(df['Property_Age'].min(),
                         df['Property_Age'].max(), 60)
dist_range = np.linspace(df['Distance_to_Tech_Hub'].min(),
                         df['Distance_to_Tech_Hub'].max(), 60)

age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Apply the regression equation at every point on the grid:
#   Ŷ = β₀ + β₁·Age + β₂·Distance
# This transforms the flat (Age, Distance) canvas into a tilted 3D surface
# whose height at each point is exactly what the model predicts.
price_grid = b0 + b1 * age_grid + b2 * dist_grid

# ── Residual Coloring for Scatter Points ─────────────────────
# Color each observed point by its residual (actual − predicted).
# Green = model under-predicted (positive residual)
# Red   = model over-predicted  (negative residual)
residuals = multi_model.resid

# ── Build the Plotly Figure ───────────────────────────────────
fig = go.Figure()

# Layer 1 — the regression hyperplane (the 2D surface in 3D space)
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid / 1e6,           # convert to $M for readability
    colorscale='Viridis',
    opacity=0.55,
    showscale=False,
    name='Regression Hyperplane',
    hovertemplate=(
        'Age: %{x:.1f} yrs<br>'
        'Distance: %{y:.1f} mi<br>'
        'Predicted: $%{z:.3f}M<extra>Hyperplane</extra>'
    )
))

# Layer 2 — observed data points, colored by residual magnitude
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'] / 1e6,
    mode='markers',
    marker=dict(
        size=3.5,
        color=residuals,           # map residual value to color
        colorscale='RdYlGn',       # red = over-predicted, green = under
        colorbar=dict(
            title=dict(text='Residual ($)', font=dict(color='white')),
            tickfont=dict(color='white'),
            x=1.02
        ),
        cmin=-residuals.abs().quantile(0.97),   # symmetric colorbar
        cmax= residuals.abs().quantile(0.97),
        opacity=0.75,
        line=dict(width=0),
    ),
    name='Observed Sales',
    hovertemplate=(
        'Age: %{x:.1f} yrs<br>'
        'Distance: %{y:.1f} mi<br>'
        'Actual: $%{z:.3f}M<extra>Observed</extra>'
    )
))

# ── Layout ────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            '<b>Multivariate OLS Regression Hyperplane</b><br>'
            '<sup>Sale Price ~ Property Age + Distance to Tech Hub  ·  '
            f'R² = {multi_model.rsquared:.4f}  ·  n = {len(df):,}</sup>'
        ),
        font=dict(color='white', size=16),
        x=0.5
    ),
    scene=dict(
        xaxis=dict(title='Property Age (years)',
                   backgroundcolor='#0d0d1a', gridcolor='#2a2a4a',
                   tickfont=dict(color='white'), title_font=dict(color='white')),
        yaxis=dict(title='Distance to Tech Hub (miles)',
                   backgroundcolor='#0d0d1a', gridcolor='#2a2a4a',
                   tickfont=dict(color='white'), title_font=dict(color='white')),
        zaxis=dict(title='Sale Price ($M)',
                   backgroundcolor='#0d0d1a', gridcolor='#2a2a4a',
                   tickfont=dict(color='white'), title_font=dict(color='white')),
        bgcolor='#0d0d1a',
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.9))   # default viewing angle
    ),
    paper_bgcolor='#0d0d1a',
    plot_bgcolor='#0d0d1a',
    font=dict(color='white'),
    margin=dict(l=0, r=0, t=90, b=0),
    height=680,
    legend=dict(
        font=dict(color='white'),
        bgcolor='rgba(255,255,255,0.05)',
        bordercolor='#444',
        borderwidth=1,
        x=0.01, y=0.99
    )
)

# ── Coefficient Annotation ────────────────────────────────────
fig.add_annotation(
    text=(
        f"<b>Regression Equation</b><br>"
        f"Ŷ = {b0/1e6:.3f}M"
        f" {b1:+,.1f}·Age"
        f" {b2:+,.1f}·Distance"
    ),
    xref='paper', yref='paper',
    x=0.01, y=0.10,
    showarrow=False,
    font=dict(color='#a5b4fc', size=11, family='monospace'),
    bgcolor='rgba(255,255,255,0.06)',
    bordercolor='#6366f1',
    borderwidth=1,
    borderpad=6,
    align='left'
)

fig.show()

Saving Zillow_California_2026_Hedonic.csv to Zillow_California_2026_Hedonic (1).csv
β₀ Intercept            : 1,202,971.27
β₁ Property_Age         :  -2,063.1292  $/yr
β₂ Distance_to_Tech_Hub :  -7,964.2448  $/mile
R²                      : 0.9543
